In [2]:
import pandas as pd
import numpy as np

# LOAD DATA & PIVOT AS BEFORE …
df = pd.read_csv('3YearData.csv', parse_dates=['Date'])
df = df.drop_duplicates(subset=['Date','Symbol'])
close_daily = df.pivot(index='Date', columns='Symbol', values='Close')
low_daily   = df.pivot(index='Date', columns='Symbol', values='Low')
all_dates   = pd.date_range(close_daily.index.min(), close_daily.index.max(), freq='B')
close_daily = close_daily.reindex(all_dates).ffill().bfill()
low_daily   = low_daily.reindex(all_dates).ffill().bfill()
weekly_close   = close_daily.resample('W-FRI').last()
weekly_returns = weekly_close.pct_change()

# STRATEGY PARAMETERS
lag             = 4
horizons        = {'r52':52, 'r26':26, 'r12':12, 'r4':4}
vol_thresh      = 0.10      # 10% 
base_num_stocks = 6
weights         = {'r52':0.4,'r26':0.3,'r12':0.2,'r4':0.1}

# COMPUTE MOMENTUM SERIES
r52 = (weekly_close.shift(lag) / weekly_close.shift(lag + horizons['r52'])) - 1
r26 = (weekly_close.shift(lag) / weekly_close.shift(lag + horizons['r26'])) - 1
r12 = (weekly_close.shift(lag) / weekly_close.shift(lag + horizons['r12'])) - 1
r4  = (weekly_close.shift(lag) / weekly_close.shift(lag + horizons['r4']))  - 1

# BUILD COMPOSITE SCORE
composite_factor = pd.DataFrame(index=weekly_close.index, columns=weekly_close.columns, dtype=float)
for date in weekly_close.index:
    data = pd.DataFrame({
        'r52': r52.loc[date],
        'r26': r26.loc[date],
        'r12': r12.loc[date],
        'r4':  r4.loc[date]
    }).dropna()
    if data.empty: continue
    z = (data - data.mean()) / data.std(ddof=0)
    comp = z['r52']*weights['r52'] + z['r26']*weights['r26'] + z['r12']*weights['r12'] + z['r4']*weights['r4']
    composite_factor.loc[date, comp.index] = comp

# BACKTEST LOOP
initial_capital = 100000.0
capital         = initial_capital
weekly_dates    = weekly_close.index
portfolio_value = pd.Series(index=weekly_dates, dtype=float)

start_offset = max(horizons.values()) + lag

for i in range(start_offset, len(weekly_dates)-1):
    today       = weekly_dates[i]
    next_friday = weekly_dates[i+1]

    # 1) 4-week drift & vol
    recent   = weekly_returns.iloc[i-3:i+1]  # last 4 weeks
    drift_4w = recent.mean(axis=1).mean()
    vol_4w   = recent.stack().std(ddof=0)

    # 2) Volatility-scaling (piecewise)
    if vol_4w < 0.08:
        exposure_scaler = 1.0
    elif vol_4w < 0.12:
        exposure_scaler = 0.75
    elif vol_4w < 0.15:
        exposure_scaler = 0.50
    else:
        exposure_scaler = 0.25
    capital_to_spend = capital * exposure_scaler
    cash_remain      = capital - capital_to_spend

    # 3) Regime filter: drift > 0.5% OR vol < 10%
    if not ((drift_4w > 0.005) or (vol_4w < vol_thresh)):
        portfolio_value[next_friday] = capital
        continue

    # 4) Two-horizon filter (adaptive thresholds)
    df_factors = pd.DataFrame({'r52': r52.loc[today], 'r12': r12.loc[today]}).dropna()
    z52 = (df_factors['r52'] - df_factors['r52'].mean()) / df_factors['r52'].std(ddof=0)
    z12 = (df_factors['r12'] - df_factors['r12'].mean()) / df_factors['r12'].std(ddof=0)
    # Adaptive cutoffs
    if drift_4w > 0.01:
        z52_thresh = 0.10
        z12_thresh = 0.02
    else:
        z52_thresh = 0.20
        z12_thresh = 0.05
    eligible = df_factors.index[(z52 > z52_thresh) | (z12 > z12_thresh)].tolist()
    if len(eligible) == 0:
        portfolio_value[next_friday] = capital
        continue

    # 5) Rank eligible by composite
    comp_today = composite_factor.loc[today, eligible].dropna()
    if comp_today.empty:
        portfolio_value[next_friday] = capital
        continue

    # 6) Flexible # of stocks: 6 ± 2, depending on drift
    if drift_4w > 0.02:
        num_stocks = min( len(eligible), 12 )
    elif drift_4w > 0.01:
        num_stocks = min( len(eligible), 8 )
    else:
        num_stocks = base_num_stocks

    ranked   = comp_today.sort_values(ascending=False).index.tolist()
    selected = ranked[:num_stocks]

    # 7) Weight by composite score
    scores_sel   = comp_today.loc[selected]
    score_sum    = scores_sel.sum()
    entry_prices = weekly_close.loc[today, selected]
    shares       = {}

    for sym in selected:
        alloc       = capital_to_spend * (scores_sel[sym] / score_sum)
        shares[sym] = alloc / entry_prices[sym]

    # 8) Volatility‐scaled trailing stop (3σ, bounded 15–30%)
    proceeds = 0.0
    for sym in selected:
        entry_price = entry_prices[sym]
        # Estimate 4-week sigma_i for this symbol
        sigma_i = weekly_returns[sym].iloc[i-3:i+1].std(ddof=0)
        stop_pct_i = np.clip(3 * sigma_i, 0.15, 0.30)
        window     = low_daily.loc[today + pd.Timedelta(days=1): next_friday, sym]
        stop_price = entry_price * (1 - stop_pct_i)
        triggered  = window[window <= stop_price]

        # 9) “Hold-through” first trading day logic:
        if not triggered.empty:
            trigger_date = triggered.index[0]
            if trigger_date <= (today + pd.Timedelta(days=1)):
                # If stop hits on the first day after entry, ignore this day’s trigger
                triggered = window[window.index > (today + pd.Timedelta(days=1))]

        if not triggered.empty:
            exit_price = triggered.iloc[0]
            proceeds  += shares[sym] * 0.25 * exit_price
            proceeds  += shares[sym] * 0.75 * weekly_close.loc[next_friday, sym]
        else:
            proceeds  += shares[sym] * weekly_close.loc[next_friday, sym]

    # 10) Combine proceeds + leftover cash
    capital = proceeds + cash_remain
    portfolio_value[next_friday] = capital

# FINAL PERFORMANCE CALCULATION
portfolio_value = portfolio_value.dropna()
rets            = portfolio_value.pct_change().fillna(0)
final_value     = portfolio_value.iloc[-1]
total_pct       = (final_value / initial_capital - 1) * 100
mean_ret        = rets.mean()
std_ret         = rets.std(ddof=0)
sharpe          = (mean_ret / std_ret) * np.sqrt(52) if std_ret != 0 else np.nan

print("Enhanced v2+ Strategy Performance:")
print(f"Final Capital: ₹{final_value:,.2f}")
print(f"Total Return: {total_pct:.2f}%")
print(f"Annualized Sharpe Ratio: {sharpe:.2f}\n")

# YEAR-BY-YEAR ANNUALIZED RETURNS
results    = []
start_date = portfolio_value.index[0]
for year in range(3):
    ps = start_date + pd.DateOffset(years=year)
    pe = ps + pd.DateOffset(years=1)

    si = portfolio_value.index.searchsorted(ps)
    if si >= len(portfolio_value):
        break
    actual_start = portfolio_value.index[si]

    ei = portfolio_value.index.searchsorted(pe)
    if ei >= len(portfolio_value) or portfolio_value.index[ei] > pe:
        ei = min(ei, len(portfolio_value) - 1)
        if portfolio_value.index[ei] > pe and ei > 0:
            ei -= 1
    actual_end = portfolio_value.index[ei]

    Vs  = portfolio_value.loc[actual_start]
    Ve  = portfolio_value.loc[actual_end]
    days = (actual_end - actual_start).days
    yrs  = days / 365.0
    ann  = (Ve / Vs) ** (1 / yrs) - 1

    results.append({
        'Year': year + 1,
        'StartDate':  actual_start.date(),
        'EndDate':    actual_end.date(),
        'AnnualizedReturnPct': ann * 100
    })

results_df = pd.DataFrame(results)
print("Year-by-Year Annualized Returns:")
print(results_df.to_markdown(index=False, floatfmt=".2f"))


Enhanced v2+ Strategy Performance:
Final Capital: ₹192,399.22
Total Return: 92.40%
Annualized Sharpe Ratio: 1.08

Year-by-Year Annualized Returns:
|   Year | StartDate   | EndDate    |   AnnualizedReturnPct |
|-------:|:------------|:-----------|----------------------:|
|      1 | 2023-02-10  | 2024-02-09 |                134.82 |
|      2 | 2024-02-16  | 2025-02-07 |                -12.95 |
|      3 | 2025-02-14  | 2025-05-30 |                 11.40 |


In [ ]:
import pandas as pd
import numpy as np

# Load the 3-year OHLC data
df = pd.read_csv('3YearData.csv', parse_dates=['Date'])
# Drop duplicates
df = df.drop_duplicates(subset=['Date', 'Symbol'], keep='first')

# Pivot daily close and low, and also prepare daily returns for volatility
close_daily = df.pivot(index='Date', columns='Symbol', values='Close')
low_daily = df.pivot(index='Date', columns='Symbol', values='Low')

# Reindex forward/backfill to business days
all_dates = pd.date_range(close_daily.index.min(), close_daily.index.max(), freq='B')
close_daily = close_daily.reindex(all_dates).ffill().bfill()
low_daily = low_daily.reindex(all_dates).ffill().bfill()

# Weekly close on Friday and weekly returns for volatility
weekly_close = close_daily.resample('W-FRI').last()
weekly_returns = weekly_close.pct_change()

# Compute 12-week volatility for scaling
vol_12w = weekly_returns.rolling(12).std(ddof=0)

# Parameters for enhanced strategy
lag = 4
horizons = {'r52': 52, 'r12': 12}
# Dynamic regime: trade if (drift > 0) or (vol < 12%)
vol_thresh = 0.12
stop_pct = 0.10  # 20% trailing stop
base_num_stocks = 6

# Compute momentum horizons (lagged)
r52 = (weekly_close.shift(lag) / weekly_close.shift(lag + horizons['r52'])) - 1
r12 = (weekly_close.shift(lag) / weekly_close.shift(lag + horizons['r12'])) - 1
r26 = (weekly_close.shift(lag) / weekly_close.shift(lag + 26)) - 1
r4  = (weekly_close.shift(lag) / weekly_close.shift(lag + 4)) - 1

# Composite factor weights
weights = {'r52': 0.3, 'r26': 0.2, 'r12': 0.25, 'r4': 0.25}

# Build composite factor DataFrame
composite_factor = pd.DataFrame(index=weekly_close.index, columns=weekly_close.columns, dtype=float)
for date in weekly_close.index:
    data = pd.DataFrame({
        'r52': r52.loc[date],
        'r26': r26.loc[date],
        'r12': r12.loc[date],
        'r4':  r4.loc[date]
    })
    data = data.dropna()
    if data.empty:
        continue
    z = (data - data.mean()) / data.std(ddof=0)
    comp_scores = z['r52']*weights['r52'] + z['r26']*weights['r26'] + z['r12']*weights['r12'] + z['r4']*weights['r4']
    composite_factor.loc[date, comp_scores.index] = comp_scores

# Backtest enhanced strategy
initial_capital = 100000.0
capital = initial_capital
weekly_dates = weekly_close.index
portfolio_value = pd.Series(index=weekly_dates, dtype=float)

for i in range(max(horizons['r52'], horizons['r12']) + lag, len(weekly_dates) - 1):
    today = weekly_dates[i]
    next_friday = weekly_dates[i + 1]

    # 4-week drift & vol
    recent = weekly_returns.iloc[i-3:i+1]
    drift_4w = recent.mean(axis=1).mean()
    vol_4w = recent.stack().std(ddof=0)

    # Regime filter: trade if drift > 0 or vol < 12%
    if not ((drift_4w > 0) or (vol_4w < vol_thresh)):
        portfolio_value[next_friday] = capital
        continue

    # Two-horizon filter: z52 > 0.1 or z12 > 0.0
    dfactors = pd.DataFrame({'r52': r52.loc[today], 'r12': r12.loc[today]}).dropna()
    if dfactors.empty:
        portfolio_value[next_friday] = capital
        continue
    z52 = (dfactors['r52'] - dfactors['r52'].mean()) / dfactors['r52'].std(ddof=0)
    z12 = (dfactors['r12'] - dfactors['r12'].mean()) / dfactors['r12'].std(ddof=0)
    eligible = dfactors.index[(z52 > 0.1) | (z12 > 0.0)].tolist()
    if len(eligible) == 0:
        portfolio_value[next_friday] = capital
        continue

    # Rank eligible by composite
    comp_scores_today = composite_factor.loc[today, eligible].dropna()
    if comp_scores_today.empty:
        portfolio_value[next_friday] = capital
        continue

    # Dynamic number: if drift > 1%, pick 10; elif > 0.5% pick 8; else 6
    if drift_4w > 0.01:
        num_stocks = 10
    elif drift_4w > 0.005:
        num_stocks = 8
    else:
        num_stocks = base_num_stocks

    ranked = comp_scores_today.sort_values(ascending=False).index.tolist()
    selected = ranked[:num_stocks] if len(ranked) >= num_stocks else ranked

    # Volatility scaling: compute 12-week vol for each at today
    vol_sel = vol_12w.loc[today, selected].dropna()
    comp_sel = comp_scores_today.loc[vol_sel.index]
    # risk weight = composite / vol
    risk_weights = comp_sel / vol_sel
    total_rw = risk_weights.sum()
    if total_rw == 0:
        portfolio_value[next_friday] = capital
        continue

    shares = {}
    proceeds = 0.0
    entry_prices = weekly_close.loc[today, selected]
    for sym in risk_weights.index:
        alloc = capital * (risk_weights[sym] / total_rw)
        shares[sym] = alloc / entry_prices[sym]

    # Trailing-stop 20%, sell 25% if hit, hold rest
    for sym in risk_weights.index:
        entry_price = entry_prices[sym]
        window = low_daily.loc[today + pd.Timedelta(days=1): next_friday, sym]
        stop_price = entry_price * (1 - stop_pct)
        triggered = window[window <= stop_price]
        if not triggered.empty:
            exit_price = triggered.iloc[0]
            proceeds += shares[sym] * 0.25 * exit_price
            proceeds += shares[sym] * 0.75 * weekly_close.loc[next_friday, sym]
        else:
            proceeds += shares[sym] * weekly_close.loc[next_friday, sym]

    capital = proceeds
    portfolio_value[next_friday] = capital

portfolio_value = portfolio_value.dropna()
returns = portfolio_value.pct_change().fillna(0)
final_value = portfolio_value.iloc[-1]
total_return_pct = (final_value / initial_capital - 1) * 100
mean_ret = returns.mean()
std_ret = returns.std(ddof=0)
sharpe = (mean_ret / std_ret) * np.sqrt(52) if std_ret != 0 else np.nan

print("Enhanced Strategy v2 Performance:")
print(f"Final Capital: ₹{final_value:,.2f}")
print(f"Total Return: {total_return_pct:.2f}%")
print(f"Annualized Sharpe: {sharpe:.2f}")

# Year-by-year
results = []
start_date = portfolio_value.index[0]
for year in range(3):
    ps = start_date + pd.DateOffset(years=year)
    pe = ps + pd.DateOffset(years=1)
    si = portfolio_value.index.searchsorted(ps)
    if si >= len(portfolio_value):
        break
    actual_start = portfolio_value.index[si]
    ei = portfolio_value.index.searchsorted(pe)
    if ei >= len(portfolio_value) or portfolio_value.index[ei] > pe:
        ei = min(ei, len(portfolio_value)-1)
        if portfolio_value.index[ei] > pe and ei > 0:
            ei -= 1
    actual_end = portfolio_value.index[ei]
    Vs = portfolio_value.loc[actual_start]
    Ve = portfolio_value.loc[actual_end]
    days = (actual_end - actual_start).days
    years_frac = days / 365.0
    annualized = (Ve / Vs) ** (1/years_frac) - 1
    results.append((year+1, actual_start.date(), actual_end.date(), annualized*100))

print("Year-by-Year Annualized Returns:")
for y, sd, ed, ar in results:
    print(f"Year {y}: {sd} to {ed} -> {ar:.2f}%")

Enhanced Strategy v2 Performance:
Final Capital: ₹193,021.00
Total Return: 93.02%
Annualized Sharpe: 1.20
Year-by-Year Annualized Returns:
Year 1: 2023-02-10 to 2024-02-09 -> 124.49%
Year 2: 2024-02-16 to 2025-02-07 -> -9.09%
Year 3: 2025-02-14 to 2025-05-30 -> 15.67%


In [7]:
import pandas as pd

# Load datasets with date parsing
df_3y = pd.read_csv('3YearData.csv', parse_dates=['Date'])
df_nifty = pd.read_csv('Nifty200.csv', parse_dates=['Date'], dayfirst=True)

# Normalize dates to date-only (remove timezone/time)
df_3y['Date'] = pd.to_datetime(df_3y['Date'].dt.date)
df_nifty['Date'] = pd.to_datetime(df_nifty['Date'].dt.date)

# drop Change % column if it exists
if 'Change %' in df_nifty.columns:
    df_nifty = df_nifty.drop(columns=['Change %'])
    
# Ensure 'Symbol' column exists in Nifty200
if 'Symbol' not in df_nifty.columns:
    df_nifty['Symbol'] = 'NIFTY200'

# Convert numeric columns
for c in ['Open','High','Low','Close']:
    df_nifty[c] = pd.to_numeric(df_nifty[c].astype(str).str.replace(',',''), errors='coerce')
 
# Parse Volume strings (“1.35B”, “950M”, or “1,234”) → float
def parse_vol(x):
    s = str(x).upper().strip().replace(',','')
    if s.endswith('B'): return float(s[:-1]) * 1e9
    if s.endswith('M'): return float(s[:-1]) * 1e6
    try:               return float(s)
    except:            return pd.NA

df_nifty['Volume'] = df_nifty['Volume'].apply(parse_vol)

# Filter Nifty200 rows to dates present in 3YearData
common_dates = df_3y['Date'].unique()
df_nifty_filtered = df_nifty[df_nifty['Date'].isin(common_dates)]

# Append Nifty200 rows to the 3YearData
appended = pd.concat([df_3y, df_nifty_filtered], ignore_index=True)

# Sort by Date then Symbol
appended = appended.sort_values(['Date', 'Symbol']).reset_index(drop=True)

# Save the final file
appended.to_csv('3YearData_appended_Nifty200.csv', index=False)

# Display Nifty rows to user
print("Nifty200 rows appended to 3YearData:")
print(appended[appended['Symbol'] == 'NIFTY200'].head(10))


Nifty200 rows appended to 3YearData:
        Symbol       Date    Close     Open     High      Low        Volume
134   NIFTY200 2022-01-03  9348.00  9236.35  9358.75  9235.10  1350000000.0
332   NIFTY200 2022-01-04  9422.45  9380.40  9432.30  9325.55  1330000000.0
530   NIFTY200 2022-01-05  9477.20  9428.45  9485.75  9391.35  1400000000.0
728   NIFTY200 2022-01-06  9405.40  9400.50  9422.40  9351.60  1240000000.0
926   NIFTY200 2022-01-07  9442.65  9432.05  9485.25  9384.25  1320000000.0
1124  NIFTY200 2022-01-10  9532.60  9489.95  9538.90  9472.95  1210000000.0
1322  NIFTY200 2022-01-11  9554.60  9530.45  9567.90  9510.85  3050000000.0
1520  NIFTY200 2022-01-12  9640.75  9613.85  9647.85  9593.00  1690000000.0
1718  NIFTY200 2022-01-13  9672.75  9665.75  9679.45  9616.25  1230000000.0
1916  NIFTY200 2022-01-14  9673.90  9637.05  9689.10  9606.60   967560000.0


In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import os

# ----------------------------------------------------------------------------------------------------------------------
# 0) CONFIGURATION
# ----------------------------------------------------------------------------------------------------------------------

# Use relative paths for better portability
DAILY_OHLC_PATH = '3YearData_appended_Nifty200.csv'
SECTOR_MAP_CSV = '/home/qa/runtime/data/historicMktData/equitySchema.csv'
MARKET_INDEX = 'NIFTY200'

# Strategy parameters (with explanations)
LAG_WEEKS = 4  # Lag all signals by 4 weeks (≈1 month) to avoid look-ahead bias
HOR_LOOKBACKS = {'r52': 52, 'r26': 26, 'r12': 12, 'r4': 4}  # Lookback periods in weeks for momentum calculation
COMPOSITE_WTS = {'r52': 0.4, 'r26': 0.3, 'r12': 0.2, 'r4': 0.1}  # Weights for composite momentum score
RESID_LOOKBACK = 52  # 1-year rolling regression window for residual momentum
RESID_MEAS_WEEKS = 12  # Sum of residuals over the past 3 months (lagged 1 month)
TSMOM_WINDOW = 12  # 12-month trailing return for time-series momentum filter
TREND_CUTOFF = 0.0  # Require 12-month return > 0 to pass trend filter
MAX_PER_SECTOR = 3  # Maximum 3 positions per sector
TOP_N = 50  # Rank top 50 by final score
VOL_ADJ_PERIOD = 12  # 12-week volatility for position sizing
MAX_POS_WEIGHTS = 0.05  # Maximum 5% of NAV per single position
TRAIL_SIGMA_MULT = 3  # 3σ trailing stop multiplier
HARDCAP_DD = 0.20  # 20% max drawdown from entry triggers full exit
REB_FREQ = 'W-FRI'  # Rebalance each Friday
initial_capital = 100000.0

# ----------------------------------------------------------------------------------------------------------------------
# 1) DATA LOADING & PRE-PROCESSING
# ----------------------------------------------------------------------------------------------------------------------

def load_data():
    """
    Load daily OHLC and sector mapping with error handling.
    Returns:
        close_df (DataFrame): daily closing prices, index=Date, columns=symbols
        low_df (DataFrame): daily low prices, index=Date, columns=symbols
        sector_map (Series): index=symbol, value=sector string
    """
    try:
        raw = pd.read_csv(DAILY_OHLC_PATH, parse_dates=['Date'])
    except FileNotFoundError:
        raise FileNotFoundError(f"Data file not found: {DAILY_OHLC_PATH}")
    
    if raw.empty:
        raise ValueError("OHLC data is empty")
    
    raw = raw.drop_duplicates(subset=['Date', 'Symbol'])
    
    # Pivot to get daily close/low
    close_df = raw.pivot(index='Date', columns='Symbol', values='Close')
    low_df = raw.pivot(index='Date', columns='Symbol', values='Low')
    
    # Reindex to business days, forward/back fill
    bizdays = pd.date_range(raw['Date'].min(), raw['Date'].max(), freq='B')
    close_df = close_df.reindex(bizdays).ffill().bfill()
    low_df = low_df.reindex(bizdays).ffill().bfill()
    
    # Exclude stocks with insufficient data or non-positive prices
    min_data_points = 252  # ~1 year of business days
    close_df = close_df.dropna(thresh=min_data_points, axis=1)
    close_df = close_df.loc[:, (close_df > 0).all()]  # Ensure positive prices
    low_df = low_df[close_df.columns]  # Align low_df to close_df columns
    
    # Verify market index exists
    if MARKET_INDEX not in close_df.columns:
        raise ValueError(f"Market index {MARKET_INDEX} not found in OHLC data")
    
    # Load sector map with explicit column selection
    try:
        sector_map = pd.read_csv(SECTOR_MAP_CSV)[['Symbol', 'Sector']].set_index('Symbol')['Sector']
    except FileNotFoundError:
        raise FileNotFoundError(f"Sector map file not found: {SECTOR_MAP_CSV}")
    
    # Align sector_map with close_df and handle missing sectors
    symbols = close_df.columns
    sector_map = sector_map.reindex(symbols, fill_value='Unknown')
    missing_sectors = sector_map[sector_map == 'Unknown'].index
    for sym in missing_sectors:
        if sym != MARKET_INDEX:
            print(f"Warning: No sector for {sym}, defaulting to 'Unknown'")
    
    return close_df, low_df, sector_map

# ----------------------------------------------------------------------------------------------------------------------
# 2) RESAMPLE TO WEEKLY PRICES (FRIDAY CLOSES) & WEEKLY RETURNS
# ----------------------------------------------------------------------------------------------------------------------

def compute_weekly(close_df, low_df):
    """
    From daily closes, get Friday closes and weekly returns, ensuring alignment with low_df.
    Returns:
        weekly_close (DataFrame): index=Friday Date, columns=symbols, values=Friday close
        weekly_ret (DataFrame): same index/columns, values=1-week pct change
        weekly_dates (Index): valid Friday dates present in both weekly_close and low_df
    """
    weekly_close = close_df.resample(REB_FREQ).last()
    weekly_ret = weekly_close.pct_change()
    if weekly_close.empty or weekly_ret.empty:
        raise ValueError("Weekly data is empty after resampling")
    
    # Ensure weekly_dates are present in low_df
    weekly_dates = weekly_close.index.intersection(low_df.index)
    if len(weekly_dates) == 0:
        raise ValueError("No common dates between weekly_close and low_df")
    
    weekly_close = weekly_close.loc[weekly_dates]
    weekly_ret = weekly_ret.loc[weekly_dates]
    
    return weekly_close, weekly_ret, weekly_dates

# ----------------------------------------------------------------------------------------------------------------------
# 3) SIGNALS CALCULATION
# ----------------------------------------------------------------------------------------------------------------------

def multi_horizon_momentum(weekly_close):
    """
    Compute cross-sectional multi-horizon momentum z-scores:
    - Calculate returns over multiple lookback periods (12m, 6m, 3m, 1m), each lagged by 4 weeks.
    - Compute z-scores for each period and a composite score.
    Returns:
        mh_zscores (dict of DataFrames): keys='r52','r26','r12','r4', values=z-scores
        composite_mh (DataFrame): weighted sum of z-scores
    """
    mh_returns = {}
    mh_zscores = {}
    for name, weeks in HOR_LOOKBACKS.items():
        ret = (weekly_close.shift(LAG_WEEKS) / weekly_close.shift(LAG_WEEKS + weeks)) - 1
        mh_returns[name] = ret
        z = (ret - ret.mean(axis=1, skipna=True)) / ret.std(axis=1, ddof=0, skipna=True)
        mh_zscores[name] = z

    composite_mh = sum(mh_zscores[name].fillna(0.0) * weight for name, weight in COMPOSITE_WTS.items())
    return mh_zscores, composite_mh

def residual_momentum(weekly_ret, weekly_close):
    """
    Compute residual momentum:
    - Run a rolling OLS (52-week window) of stock_ret ~ market_ret.
    - Calculate residual returns and sum over the past 12 weeks (lagged 4 weeks).
    Returns:
        resid_sum (DataFrame): sum of residuals over the measurement period
    """
    if MARKET_INDEX not in weekly_ret.columns:
        raise ValueError(f"Market index {MARKET_INDEX} not found in weekly returns")
    
    symbols = [s for s in weekly_close.columns if s != MARKET_INDEX]
    if not symbols:
        raise ValueError("No valid symbols found for residual momentum calculation")
    
    market = weekly_ret[MARKET_INDEX].values.reshape(-1, 1)
    n_weeks = len(weekly_ret)

    resid_sum = pd.DataFrame(np.nan, index=weekly_ret.index, columns=symbols)
    betas = pd.DataFrame(index=weekly_ret.index, columns=symbols)
    
    for sym in symbols:
        Y = weekly_ret[sym].values.reshape(-1, 1)
        for t in range(RESID_LOOKBACK, n_weeks):
            Xwin = market[t - RESID_LOOKBACK:t]
            Ywin = Y[t - RESID_LOOKBACK:t]
            if len(Xwin) < RESID_LOOKBACK or np.isnan(Xwin).any() or np.isnan(Ywin).any():
                continue
            lr = LinearRegression().fit(Xwin, Ywin)
            betas.at[weekly_ret.index[t], sym] = lr.coef_[0, 0]

    resid = weekly_ret[symbols] - betas * weekly_ret[MARKET_INDEX].values.reshape(-1, 1)
    for t in range(RESID_LOOKBACK + RESID_MEAS_WEEKS + LAG_WEEKS - 1, n_weeks):
        start = t - LAG_WEEKS - RESID_MEAS_WEEKS + 1
        end = t - LAG_WEEKS
        resid_sum.iloc[t] = resid.iloc[start:end + 1].sum(axis=0)

    return resid_sum

def time_series_filter(weekly_close):
    """
    Compute each stock’s 12-month trailing return (lagged by 4 weeks).
    Return a boolean DataFrame: True if trailing return > TREND_CUTOFF, else False.
    """
    ts_ret = (weekly_close.shift(LAG_WEEKS) / weekly_close.shift(LAG_WEEKS + TSMOM_WINDOW)) - 1
    ts_filter = (ts_ret > TREND_CUTOFF) & ts_ret.notna()
    return ts_filter

# ----------------------------------------------------------------------------------------------------------------------
# 4) RANKING & SELECTION
# ----------------------------------------------------------------------------------------------------------------------

def rank_and_select(friday, composite_mh, resid_sum, ts_filter, sector_map):
    """
    For a given Friday, select top stocks based on final score:
    - Apply time-series filter, excluding market index.
    - Compute final_score = 0.7*composite_mh + 0.3*resid_zscore.
    - Rank descending and enforce sector limits.
    Returns:
        list of selected symbols
    """
    eligible = ts_filter.loc[friday][ts_filter.loc[friday] == True].index.tolist()
    eligible = [sym for sym in eligible if sym != MARKET_INDEX]
    resid_vals = resid_sum.loc[friday, [s for s in eligible if s in resid_sum.columns]].dropna()
    
    if resid_vals.empty:
        print(f"Warning: No valid residual values for {friday}, returning empty selection")
        return []
    
    z_resid = (resid_vals - resid_vals.mean()) / resid_vals.std(ddof=0) if not resid_vals.empty else pd.Series()
    
    data = []
    for sym in resid_vals.index:
        if sym in composite_mh.columns:
            mh_score = composite_mh.at[friday, sym]
            res_score = z_resid.get(sym, 0.0)
            final = 0.7 * mh_score + 0.3 * res_score
            data.append((sym, final))
    
    if not data:
        print(f"Warning: No valid scores for {friday}, returning empty selection")
        return []
    
    scored = pd.DataFrame(data, columns=['Symbol', 'Score']).set_index('Symbol').sort_values('Score', ascending=False)
    
    selections = []
    sector_counts = {}
    for sym in scored.index:
        sector = sector_map.get(sym, 'Unknown')
        cnt = sector_counts.get(sector, 0)
        if cnt < MAX_PER_SECTOR:
            selections.append(sym)
            sector_counts[sector] = cnt + 1
        if len(selections) >= TOP_N:
            break
    return selections

# ----------------------------------------------------------------------------------------------------------------------
# 5) POSITION SIZING (VOLATILITY ADJUSTED)
# ----------------------------------------------------------------------------------------------------------------------

def vol_adjusted_sizing(weekly_close, holdings, friday, capital, weekly_dates):
    """
    Size positions based on trailing volatility:
    - Compute 12-week volatility for each stock.
    - Assign weights inversely proportional to volatility, capped at MAX_POS_WEIGHTS.
    Returns:
        dict {symbol: target_value}
    """
    if not holdings:
        return {}
    
    vols = {}
    idx = weekly_dates.get_loc(friday)
    for sym in holdings:
        hist = weekly_close[sym].pct_change().iloc[max(0, idx - VOL_ADJ_PERIOD + 1):idx + 1]
        if len(hist) >= VOL_ADJ_PERIOD:
            sigma = hist.std(ddof=0)
        else:
            sigma = 1e-6
        vols[sym] = sigma if sigma > 0 else 1e-6
    
    inv_vol = {sym: 1.0 / vol for sym, vol in vols.items()}
    total_inv = sum(inv_vol.values()) or 1e-6  # Avoid division by zero
    raw_wts = {sym: inv_vol[sym] / total_inv for sym in holdings}
    
    capped = {}
    excess = 0.0
    for sym, w in raw_wts.items():
        if w > MAX_POS_WEIGHTS:
            capped[sym] = MAX_POS_WEIGHTS
            excess += (w - MAX_POS_WEIGHTS)
        else:
            capped[sym] = w
    
    if excess > 0.0:
        uncapped = [s for s in holdings if raw_wts[s] <= MAX_POS_WEIGHTS]
        total_uncapped = sum(raw_wts[s] for s in uncapped) or 1e-6
        for s in uncapped:
            delta = (raw_wts[s] / total_uncapped) * excess
            capped[s] += delta
    
    targets = {sym: capped[sym] * capital for sym in holdings}
    return targets

# ----------------------------------------------------------------------------------------------------------------------
# 6) TRAILING STOP / RISK MANAGEMENT
# ----------------------------------------------------------------------------------------------------------------------

class Position:
    def __init__(self, symbol, entry_price, entry_date, target_shares):
        self.symbol = symbol
        self.entry_price = entry_price
        self.entry_date = entry_date
        self.shares = target_shares
        self.remaining = target_shares
        self.full_exit = False

    def check_trailing_stop(self, low_series, current_date, weekly_close, weekly_dates):
        """
        Check for trailing stop or hard-cap drawdown triggers using daily lows.
        Returns proceeds from any partial or complete exit.
        """
        sym = self.symbol
        try:
            entry_idx = low_series.index.get_loc(self.entry_date)
        except KeyError:
            print(f"Warning: Entry date {self.entry_date} not in low_series for {sym}, skipping stop check")
            return 0.0
        
        # Find the nearest date <= current_date if exact match is missing
        if current_date not in low_series.index:
            valid_dates = low_series.index[low_series.index <= current_date]
            if not valid_dates.empty:
                current_date = valid_dates[-1]  # Use the latest available date
            else:
                print(f"Warning: No valid date <= {current_date} for {sym}, skipping stop check")
                return 0.0
        
        current_idx = low_series.index.get_loc(current_date)
        
        hist_ret = weekly_close[sym].pct_change().iloc[:weekly_dates.get_loc(current_date) + 1]
        sigma_i = hist_ret.tail(VOL_ADJ_PERIOD).std(ddof=0) if len(hist_ret) >= VOL_ADJ_PERIOD else 1e-6
        stop_pct_i = np.clip(TRAIL_SIGMA_MULT * sigma_i, 0.15, 0.30)
        stop_price = self.entry_price * (1 - stop_pct_i)
        
        proceeds = 0.0
        for day in low_series.index[entry_idx + 1:current_idx + 1]:
            low_price = low_series.at[day]
            if low_price <= stop_price and day > self.entry_date + pd.Timedelta(days=1):
                half_shares = self.remaining * 0.5
                proceeds += half_shares * low_price
                self.remaining -= half_shares
                break
        
        current_low = low_series.at[current_date]
        if (current_low / self.entry_price) - 1 <= -HARDCAP_DD and not self.full_exit:
            proceeds += self.remaining * current_low
            self.remaining = 0
            self.full_exit = True
        
        return proceeds

# ----------------------------------------------------------------------------------------------------------------------
# 7) MAIN BACKTEST / REBALANCE ENGINE
# ----------------------------------------------------------------------------------------------------------------------

def backtest():
    """
    Main backtest loop:
    - Compute signals, select holdings, size positions, and manage risk.
    Returns:
        portfolio_value (Series): NAV at each Friday close
    """
    close_df, low_df, sector_map = load_data()
    weekly_close, weekly_ret, weekly_dates = compute_weekly(close_df, low_df)
    
    mh_zscores, composite_mh = multi_horizon_momentum(weekly_close)
    resid_sum = residual_momentum(weekly_ret, weekly_close)
    ts_filter = time_series_filter(weekly_close)
    
    start_offset = max(RESID_LOOKBACK + RESID_MEAS_WEEKS + LAG_WEEKS, TSMOM_WINDOW + LAG_WEEKS)
    if start_offset >= len(weekly_dates):
        raise ValueError("Insufficient data for backtest")
    
    live_positions = {}
    portfolio_series = pd.Series(dtype=float, index=weekly_dates[start_offset:])
    cash = initial_capital
    
    for friday in weekly_dates[start_offset:]:
        for sym, pos in list(live_positions.items()):
            if pos.entry_date < friday:
                proceeds = pos.check_trailing_stop(low_df[sym], friday, weekly_close, weekly_dates)
                cash += proceeds
                if pos.remaining == 0:
                    del live_positions[sym]
        
        eq_value = sum(pos.remaining * weekly_close.at[friday, sym] for sym, pos in live_positions.items())
        portfolio_value = eq_value + cash
        
        selected = rank_and_select(friday, composite_mh, resid_sum, ts_filter, sector_map)
        targets_dollar = vol_adjusted_sizing(weekly_close, selected, friday, portfolio_value, weekly_dates)
        
        for sym in selected:
            target_val = targets_dollar.get(sym, 0.0)
            current_shares = live_positions[sym].remaining if sym in live_positions else 0.0
            current_val = current_shares * weekly_close.at[friday, sym]
            delta_val = target_val - current_val
            
            price = weekly_close.at[friday, sym]
            if np.isnan(price) or price <= 0:
                print(f"Warning: Invalid price for {sym} on {friday}, skipping")
                continue
            if delta_val > 0:
                new_shares = delta_val / price
                cash -= delta_val
                if sym in live_positions:
                    live_positions[sym].remaining += new_shares
                else:
                    live_positions[sym] = Position(sym, price, friday, new_shares)
            elif delta_val < 0:
                sell_shares = min(-delta_val / price, current_shares)
                proceeds = sell_shares * price
                cash += proceeds
                live_positions[sym].remaining -= sell_shares
                if live_positions[sym].remaining <= 0:
                    del live_positions[sym]
        
        eq_value_post = sum(pos.remaining * weekly_close.at[friday, sym] for sym, pos in live_positions.items())
        portfolio_series.at[friday] = eq_value_post + cash
    
    return portfolio_series

# ----------------------------------------------------------------------------------------------------------------------
# 8) RUN BACKTEST & OUTPUT PERFORMANCE
# ----------------------------------------------------------------------------------------------------------------------

if __name__ == "__main__":
    pv = backtest()
    returns = pv.pct_change().dropna()
    final_val = pv.iloc[-1]
    total_ret = (final_val / initial_capital - 1) * 100
    mean_r = returns.mean()
    std_r = returns.std(ddof=0)
    sharpe = (mean_r / std_r) * np.sqrt(52) if std_r != 0 else np.nan

    print("Blended Momentum (Enhanced v2+) Backtest Results")
    print(f"Final NAV: ₹{final_val:,.2f}")
    print(f"Total Return (3 Yrs): {total_ret:.2f} %")
    print(f"Annualized Sharpe: {sharpe:.2f}\n")

    results = []
    dates = pv.index
    for yr in range(3):
        start = dates[0] + pd.DateOffset(years=yr)
        end = start + pd.DateOffset(years=1)
        si = dates.searchsorted(start)
        ei = dates.searchsorted(end)
        if si >= len(dates):
            break
        if ei >= len(dates) or dates[ei] > end:
            ei = min(ei, len(dates) - 1)
            if dates[ei] > end and ei > 0:
                ei -= 1
        vs = pv.iloc[si]
        ve = pv.iloc[ei]
        days = (dates[ei] - dates[si]).days
        yrs = days / 365.0
        yoy = (ve / vs) ** (1 / yrs) - 1 if yrs > 0 else 0
        results.append({
            'Year': yr + 1,
            'Start': dates[si].date(),
            'End': dates[ei].date(),
            'AnnReturn(%)': yoy * 100
        })
    res_df = pd.DataFrame(results)
    print("Yearly Performance:")
    print(res_df.to_markdown(index=False, floatfmt=".2f"))

Blended Momentum (Enhanced v2+) Backtest Results
Final NAV: ₹115,129.38
Total Return (3 Yrs): 15.13 %
Annualized Sharpe: 0.51

Yearly Performance:
|   Year | Start      | End        |   AnnReturn(%) |
|-------:|:-----------|:-----------|---------------:|
|      1 | 2023-04-28 | 2024-04-26 |         237.30 |
|      2 | 2024-05-03 | 2025-04-25 |         -70.38 |
|      3 | 2025-05-02 | 2025-05-23 |         424.81 |


In [ ]:
import pandas as pd
import numpy as np

# 1) Load daily data
df = pd.read_csv('/mnt/data/3YearData.csv', parse_dates=['Date'])
df = df.drop_duplicates(subset=['Date', 'Symbol'], keep='first')

# 2) Pivot daily close/low, reindex to business days
close_daily = df.pivot(index='Date', columns='Symbol', values='Close')
low_daily   = df.pivot(index='Date', columns='Symbol', values='Low')
all_days    = pd.date_range(close_daily.index.min(), close_daily.index.max(), freq='B')
close_daily = close_daily.reindex(all_days).ffill().bfill()
low_daily   = low_daily.reindex(all_days).ffill().bfill()

# 3) Weekly Friday close & returns
weekly_close = close_daily.resample('W-FRI').last()
weekly_ret   = weekly_close.pct_change()

# 4) Sector map
schema     = pd.read_csv('/mnt/data/equitySchema.csv')
sector_map = pd.Series(schema['Sector'].values, index=schema['Symbol']).to_dict()

# 5) Multi-horizon composite momentum
lag      = 4
horizons = [52, 26, 12, 4]
weights  = [0.4, 0.3, 0.2, 0.1]
symbols  = weekly_close.columns.tolist()

mh_scores = pd.DataFrame(0.0, index=weekly_close.index, columns=symbols)
for w, h in zip(weights, horizons):
    ret = weekly_close.shift(lag) / weekly_close.shift(lag + h) - 1
    z   = ret.sub(ret.mean(axis=1), axis=0).div(ret.std(axis=1, ddof=0), axis=0)
    mh_scores += w * z.fillna(0)

# 6) Time-series filter: require 12m return lagged by 4w > 0
ts_ret    = weekly_close.shift(lag) / weekly_close.shift(lag + 12) - 1
ts_filter = ts_ret > 0

# 7) Backtest parameters
initial_capital = 100_000.0
capital         = initial_capital
cash            = capital
live_positions  = {}  # sym -> {'shares', 'entry'}
weekly_dates    = weekly_close.index
nav             = pd.Series(index=weekly_dates, dtype=float)
top_n           = 50
max_per_sec     = 3
vol_period      = 12
max_pos_wt      = 0.05
stop_pct_flat   = 0.20

start_i = lag + max(horizons)

for i in range(start_i, len(weekly_dates)):
    friday = weekly_dates[i]
    # 7A) apply trailing stops (full exit if low <= entry*(1-stop_pct_flat))
    for sym, pos in list(live_positions.items()):
        window = low_daily[sym].loc[weekly_dates[i-1] + pd.Timedelta(days=1): friday]
        if not window.empty and (window <= pos['entry'] * (1 - stop_pct_flat)).any():
            # exit all
            cash += pos['shares'] * window.min()
            del live_positions[sym]
    # 7B) record NAV before rebalance
    eq = sum(pos['shares'] * weekly_close.at[friday, sym] for sym, pos in live_positions.items())
    nav.at[friday] = cash + eq
    # 7C) select universe passing TS filter
    eligible = [s for s in symbols if ts_filter.at[friday, s]]
    # 7D) rank by mh_scores
    scores = mh_scores.loc[friday, eligible].dropna().sort_values(ascending=False)
    # 7E) sector cap
    sel   = []
    counts = {}
    for sym in scores.index:
        sec = sector_map.get(sym)
        if counts.get(sec, 0) < max_per_sec:
            sel.append(sym)
            counts[sec] = counts.get(sec, 0) + 1
        if len(sel) >= top_n:
            break
    # 7F) position sizing: inverse-vol 12w, cap 5%
    vol = weekly_ret[sel].rolling(vol_period).std(ddof=0).iloc[i]
    inv = (1 / vol).fillna(0)
    wts = inv / inv.sum() if inv.sum() > 0 else inv
    wts = wts.clip(upper=max_pos_wt)
    wts = wts / wts.sum() if wts.sum() > 0 else wts
    target_vals = wts * nav.at[friday]
    # 7G) rebalance: sell holdings not in sel
    for sym in list(live_positions):
        if sym not in sel:
            cash += live_positions[sym]['shares'] * weekly_close.at[friday, sym]
            del live_positions[sym]
    # buy/sell to targets
    for sym in sel:
        price = weekly_close.at[friday, sym]
        targ_val = target_vals.get(sym, 0.0)
        targ_shares = targ_val / price if price > 0 else 0.0
        cur_shares = live_positions.get(sym, {}).get('shares', 0.0)
        delta = targ_shares - cur_shares
        if delta > 0:
            cost = delta * price
            cash -= cost
            live_positions[sym] = {'shares': cur_shares + delta, 'entry': price}
        elif delta < 0:
            cash += (-delta) * price
            live_positions[sym]['shares'] = cur_shares + delta
    # 7H) record NAV after rebalance
    eq = sum(pos['shares'] * weekly_close.at[friday, sym] for sym, pos in live_positions.items())
    nav.at[friday] = cash + eq

# 8) performance
nav = nav.dropna()
rets = nav.pct_change().dropna()
final_val = nav.iloc[-1]
total_ret = (final_val / initial_capital - 1) * 100
sharpe = rets.mean() / rets.std(ddof=0) * np.sqrt(52)

print(f"Final NAV: ₹{final_val:,.2f}")
print(f"Total Return: {total_ret:.2f}%")
print(f"Annualized Sharpe: {sharpe:.2f}\n")

# Yearly breakdown
years = []
for y in range(3):
    start = nav.index.searchsorted(nav.index[0] + pd.DateOffset(years=y))
    end   = nav.index.searchsorted(nav.index[0] + pd.DateOffset(years=y+1))
    if start >= len(nav): break
    vs = nav.iloc[start]
    ve = nav.iloc[end-1 if end-1 < len(nav) else -1]
    days = (nav.index[end-1 if end-1 < len(nav) else -1] - nav.index[start]).days
    yrs  = days / 365.0
    yr_ret = (ve / vs) ** (1 / yrs) - 1
    years.append({'Year': y+1, 'Start': nav.index[start].date(), 'End': nav.index[end-1].date(), 'AnnReturn(%)': yr_ret*100})
print(pd.DataFrame(years).to_markdown(index=False, floatfmt=".2f"))
